### # Retriever using Faiss db (for understanding)

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [3]:
pdf_loader = PyPDFLoader(file_path='./docs/PAPER1.pdf', mode="page")

all_docs = pdf_loader.load()

In [4]:
rcts_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

rcts_docs = rcts_splitter.split_documents(all_docs)

len(rcts_docs)

49

In [5]:
embed_model = 'mxbai-embed-large:latest'

ollama_embed = OllamaEmbeddings(model=embed_model)
ollama_embed

OllamaEmbeddings(model='mxbai-embed-large:latest', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

### Parameters Explained

- **`search_type="mmr"`**: This specifies the search algorithm to use. **MMR** stands for **Maximal Marginal Relevance**, which balances relevance with diversity. It retrieves documents that are not only relevant to the query but also dissimilar from one another, ensuring a broader range of information.

- **`search_kwargs`**: This dictionary holds additional parameters for the search.

  - **`"k": 6`**: This indicates that the retrieval process should return the top **6** most relevant documents. This is the number of results you want to retrieve based on the query.

  - **`"lambda_mult": 0.25`**: This parameter adjusts the balance between relevance and diversity in the MMR algorithm. A **lambda_mult** value of **0.25** suggests a weighting that favors relevance even more than diversity, but still allows for some level of variation in the results.

### Summary

Overall, this snippet sets up a retriever that will fetch 6 documents that are not only relevant to the given query but also diverse enough to provide different perspectives, leveraging the MMR algorithm to manage the balance between these two aspects effectively.

In [6]:
faiss_db = FAISS.from_documents(rcts_docs, ollama_embed)
retriever = faiss_db.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25})

retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002BB589AF0B0>, search_type='mmr', search_kwargs={'k': 6, 'lambda_mult': 0.25})

In [7]:
retriever.invoke('what is q k v?')

[Document(id='57b89c07-6420-4683-b814-13e8947ba0ac', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './docs/PAPER1.pdf', 'total_pages': 15, 'page': 3, 'page_label': '4'}, page_content='extremely small gradients 4. To counteract this effect, we scale the dot products by 1√dk\n.\n3.2.2 Multi-Head Attention\nInstead of performing a single attention function with dmodel-dimensional keys, values and queries,\nwe found it beneficial to linearly project the queries, keys and values h times with different, learned\nlinear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of\nqueries, keys and values we then perform the attention function in parallel

## Parameters for the `as_retriever` Method

Here’s a list of parameters you can use with the `as_retriever` method in LangChain, which configures how the retriever interacts with the vector store:

### Main Parameter
- **`search_type`**: Defines the type of search to perform.
  - Options include:
    - `'similarity'` (default)
    - `'mmr'` (Maximal Marginal Relevance)
    - `'similarity_score_threshold'`

### Search Keyword Arguments (`search_kwargs`)
- **`k`**: Number of documents to return. 
  - Default: **4**
  
- **`score_threshold`**: Minimum relevance threshold for the search when using the `similarity_score_threshold` method.
  
- **`fetch_k`**: Number of documents to initially consider for MMR processing.
  - Default: **20**
  
- **`lambda_mult`**: Controls the diversity of results returned by MMR. 
  - Values range from **0** (max diversity) to **1** (min diversity).
  - Default: **0.5**
  
- **`filter`**: Criteria for filtering the results based on document metadata (e.g., only fetch documents from a certain paper).

### Summary of Examples
1. **Diverse Result Retrieval**:
   ```python
   docsearch.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25})
   ```

2. **Increased Candidate Documents for MMR**:
   ```python
   docsearch.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 50})
   ```

3. **Relevance Score Filtering**:
   ```python
   docsearch.as_retriever(search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.8})
   ```

4. **Single Most Similar Document**:
   ```python
   docsearch.as_retriever(search_kwargs={"k": 1})
   ```

5. **Metadata Filtering**:
   ```python
   docsearch.as_retriever(search_kwargs={"filter": {"paper_title": "GPT-4 Technical Report"}})
   ```

These parameters provide flexibility in how the retriever interacts with the vector database, allowing for more customized and effective retrieval strategies based on specific application needs.